# 04 - AlphaEval 因子筛选
评价因子的预测能力、滚动稳定性、扰动鲁棒性、公式逻辑、RRE 与 DPP 多样性。完整说明见 `docs/运行手册.md`。

In [ ]:
from pathlib import Path
import os
if not Path('configs/training_config.yaml').exists():
    repo=os.environ.get('ALPHAMINING_REPO_URL','https://github.com/JacksonChiy/AlphaMining-GFlowNet-AlphaEval.git')
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA 状态:',torch.cuda.is_available(),'CUDA 运行时:',torch.version.cuda)
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0); print('GPU 型号:',p.name); print('GPU 显存（GB）:',round(p.total_memory/1024**3,2))
else: print('GPU 不可用；GPU 显存（GB）: 0')

In [ ]:
import yaml,pandas as pd
config=yaml.safe_load(open('configs/training_config.yaml',encoding='utf-8'))
from src.alpha_eval import AlphaEval,AlphaEvalConfig
from src.utils import slice_date_range
cfg=dict(config['alpha_eval']); cfg['horizon']=config['dataset']['horizon']
price=slice_date_range(pd.read_pickle(config['dataset']['output']),config['dataset'].get('mining_start_date'),config['dataset'].get('mining_end_date'),label='AlphaEval price')
factors=slice_date_range(pd.read_pickle('results/alpha_factor_matrix.pkl'),config['dataset'].get('mining_start_date'),config['dataset'].get('mining_end_date'),label='AlphaEval factors')
result=AlphaEval(price,factors,AlphaEvalConfig(**cfg)).evaluate(pd.read_csv('results/alpha_pool.csv'))
display(result.head(30))